In [1]:
!pip install torchinfo

from torchinfo import summary as torchinfo_summary
!pip install torchinfo

In [2]:
"""WaveFusion-Net: Dual-Branch Image Deblurring with Wavelet-Spatial Fusion

RealBlur-J training script (NO test leakage)

Key rule:
  - TEST split is used only once at the end for final reporting.
  - Model selection ("best checkpoint") is done using a VAL split created
	deterministically from the TRAIN split.

This file intentionally keeps the *model architecture* identical to your
WaveFusion-Net notebooks. Only the dataset split/evaluation protocol is made
methodologically correct.
"""

import os
import json
import time
import math
import random
import gc
import hashlib
from glob import glob

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models

try:
	from torchinfo import summary as torchinfo_summary
except ImportError:
	torchinfo_summary = None

import matplotlib.pyplot as plt


def _has_realblur_structure(realblur_root: str) -> bool:
	"""Returns True if `realblur_root` looks like .../RealBlur with train/test folders."""
	train_blur = os.path.join(realblur_root, "train", "blur")
	train_gt = os.path.join(realblur_root, "train", "gt")
	test_blur = os.path.join(realblur_root, "test", "blur")
	test_gt = os.path.join(realblur_root, "test", "gt")
	if not (os.path.isdir(train_blur) and os.path.isdir(train_gt) and os.path.isdir(test_blur) and os.path.isdir(test_gt)):
		return False
	# Require at least one file to avoid false positives
	for ext in ("png", "jpg", "jpeg", "PNG", "JPG", "JPEG"):
		if glob(os.path.join(train_blur, f"*.{ext}")):
			return True
	return False


def resolve_realblur_root(preferred: str) -> str:
	"""Resolve the RealBlur-J root path in Kaggle.

	Many Kaggle notebooks mount datasets under different folder names.
	This function:
	- Uses `preferred` if it looks valid
	- Otherwise searches `/kaggle/input/**/RealBlur` for a matching structure
	"""
	if preferred and _has_realblur_structure(preferred):
		return preferred

	search_base = "/kaggle/input"
	if not os.path.isdir(search_base):
		return preferred  # non-Kaggle environment

	patterns = [
		os.path.join(search_base, "**", "RealBlur", "train", "blur", "*.png"),
		os.path.join(search_base, "**", "RealBlur", "train", "blur", "*.jpg"),
		os.path.join(search_base, "**", "RealBlur", "train", "blur", "*.jpeg"),
	]
	for pat in patterns:
		matches = glob(pat, recursive=True)
		if matches:
			m = matches[0]
			realblur_root = os.path.dirname(os.path.dirname(os.path.dirname(m)))
			if _has_realblur_structure(realblur_root):
				return realblur_root

	return preferred


def seed_everything(seed: int) -> None:
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)


def sha256_of_list(lines: list[str]) -> str:
	h = hashlib.sha256()
	for line in lines:
		h.update((line + "\n").encode("utf-8"))
	return h.hexdigest()


def export_split_proof(
	save_dir: str,
	train_relpaths: list[str],
	val_relpaths: list[str],
	test_relpaths: list[str],
	config: dict,
) -> str:
	"""Exports a compact proof artifact showing what was used for train/val/test.

	We store counts + SHA256 hashes of sorted file lists. This is small but
	strong evidence of the exact split used.
	"""
	os.makedirs(save_dir, exist_ok=True)

	train_sorted = sorted(train_relpaths)
	val_sorted = sorted(val_relpaths)
	test_sorted = sorted(test_relpaths)

	proof = {
		"created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
		"val_ratio": config.get("val_ratio"),
		"split_seed": config.get("split_seed"),
		"counts": {
			"train": len(train_sorted),
			"val": len(val_sorted),
			"test": len(test_sorted),
		},
		"sha256": {
			"train": sha256_of_list(train_sorted),
			"val": sha256_of_list(val_sorted),
			"test": sha256_of_list(test_sorted),
		},
		"samples": {
			"train_first10": train_sorted[:10],
			"val_first10": val_sorted[:10],
			"test_first10": test_sorted[:10],
		},
		"notes": {
			"protocol": "best checkpoint selected on VAL-from-TRAIN; TEST used only once at end",
		},
	}

	out_path = os.path.join(save_dir, "split_proof_realblurj.json")
	with open(out_path, "w", encoding="utf-8") as f:
		json.dump(proof, f, indent=2)

	return out_path


# ======================== WAVELET TRANSFORMS ========================
class DWT(nn.Module):
	"""Discrete Wavelet Transform using Haar wavelets"""

	def __init__(self):
		super().__init__()
		self.register_buffer("ll", torch.tensor([[0.5, 0.5], [0.5, 0.5]]).view(1, 1, 2, 2))
		self.register_buffer("lh", torch.tensor([[0.5, 0.5], [-0.5, -0.5]]).view(1, 1, 2, 2))
		self.register_buffer("hl", torch.tensor([[0.5, -0.5], [0.5, -0.5]]).view(1, 1, 2, 2))
		self.register_buffer("hh", torch.tensor([[0.5, -0.5], [-0.5, 0.5]]).view(1, 1, 2, 2))

	def forward(self, x):
		B, C, H, W = x.shape
		if H % 2 != 0:
			x = F.pad(x, (0, 0, 0, 1))
		if W % 2 != 0:
			x = F.pad(x, (0, 1, 0, 0))

		x = x.contiguous().view(B * C, 1, x.shape[2], x.shape[3])
		ll = F.conv2d(x, self.ll, stride=2)
		lh = F.conv2d(x, self.lh, stride=2)
		hl = F.conv2d(x, self.hl, stride=2)
		hh = F.conv2d(x, self.hh, stride=2)

		ll = ll.view(B, C, ll.shape[2], ll.shape[3])
		lh = lh.view(B, C, lh.shape[2], lh.shape[3])
		hl = hl.view(B, C, hl.shape[2], hl.shape[3])
		hh = hh.view(B, C, hh.shape[2], hh.shape[3])
		return ll, lh, hl, hh


class IDWT(nn.Module):
	"""Inverse Discrete Wavelet Transform"""

	def __init__(self):
		super().__init__()
		self.register_buffer("ll", torch.tensor([[0.5, 0.5], [0.5, 0.5]]).view(1, 1, 2, 2))
		self.register_buffer("lh", torch.tensor([[0.5, 0.5], [-0.5, -0.5]]).view(1, 1, 2, 2))
		self.register_buffer("hl", torch.tensor([[0.5, -0.5], [0.5, -0.5]]).view(1, 1, 2, 2))
		self.register_buffer("hh", torch.tensor([[0.5, -0.5], [-0.5, 0.5]]).view(1, 1, 2, 2))

	def forward(self, ll, lh, hl, hh):
		B, C, H, W = ll.shape
		ll = ll.view(B * C, 1, H, W)
		lh = lh.view(B * C, 1, H, W)
		hl = hl.view(B * C, 1, H, W)
		hh = hh.view(B * C, 1, H, W)

		ll = F.conv_transpose2d(ll, self.ll, stride=2)
		lh = F.conv_transpose2d(lh, self.lh, stride=2)
		hl = F.conv_transpose2d(hl, self.hl, stride=2)
		hh = F.conv_transpose2d(hh, self.hh, stride=2)

		out = (ll + lh + hl + hh).view(B, C, ll.shape[2], ll.shape[3])
		return out


# ======================== ATTENTION MODULES ========================
class SimpleGate(nn.Module):
	"""Simple gating mechanism - splits channels and applies element-wise product"""

	def forward(self, x):
		x1, x2 = x.chunk(2, dim=1)
		return x1 * x2


class StripAttention(nn.Module):
	"""Strip attention - processes horizontal and vertical strips separately"""

	def __init__(self, channels, strip_size=7):
		super().__init__()
		self.h_conv = nn.Conv2d(channels, channels, (1, strip_size), padding=(0, strip_size // 2), groups=channels)
		self.v_conv = nn.Conv2d(channels, channels, (strip_size, 1), padding=(strip_size // 2, 0), groups=channels)
		self.proj = nn.Conv2d(channels * 2, channels, 1)

	def forward(self, x):
		B, C, H, W = x.shape
		h_out = self.h_conv(x)[:, :, :H, :W]
		v_out = self.v_conv(x)[:, :, :H, :W]
		h_attn = torch.sigmoid(h_out)
		v_attn = torch.sigmoid(v_out)
		combined = torch.cat([x * h_attn, x * v_attn], dim=1)
		return self.proj(combined)


class SCA(nn.Module):
	"""Simplified Channel Attention"""

	def __init__(self, channels):
		super().__init__()
		self.gap = nn.AdaptiveAvgPool2d(1)
		self.fc = nn.Conv2d(channels, channels, 1)

	def forward(self, x):
		return x * self.fc(self.gap(x))


# ======================== CORE BLOCKS ========================
class NAFBlock(nn.Module):
	"""NAFNet-style block with simplified attention"""

	def __init__(self, channels, dw_expand=2):
		super().__init__()
		dw_channels = channels * dw_expand

		self.conv1 = nn.Conv2d(channels, dw_channels, 1)
		self.conv2 = nn.Conv2d(dw_channels, dw_channels, 3, padding=1, groups=dw_channels)
		self.conv3 = nn.Conv2d(dw_channels // 2, channels, 1)

		self.sca = SCA(dw_channels // 2)
		self.sg = SimpleGate()

		self.norm = nn.LayerNorm(channels)

	def forward(self, x):
		residual = x

		x = x.permute(0, 2, 3, 1)
		x = self.norm(x)
		x = x.permute(0, 3, 1, 2)

		x = self.conv1(x)
		x = self.conv2(x)
		x = self.sg(x)
		x = self.sca(x)
		x = self.conv3(x)

		return x + residual


class WaveletBlock(nn.Module):
	"""Process wavelet coefficients with attention to high-frequency details"""

	def __init__(self, channels):
		super().__init__()
		self.ll_conv = nn.Sequential(
			nn.Conv2d(channels, channels, 3, padding=1),
			nn.GELU(),
			nn.Conv2d(channels, channels, 3, padding=1),
		)
		self.hf_conv = nn.Sequential(
			nn.Conv2d(channels * 3, channels * 3, 3, padding=1, groups=3),
			nn.GELU(),
			nn.Conv2d(channels * 3, channels * 3, 3, padding=1, groups=3),
		)
		self.hf_attn = nn.Sequential(
			nn.AdaptiveAvgPool2d(1),
			nn.Conv2d(channels * 3, channels * 3, 1),
			nn.Sigmoid(),
		)

	def forward(self, ll, lh, hl, hh):
		ll_out = ll + self.ll_conv(ll)

		hf = torch.cat([lh, hl, hh], dim=1)
		hf_feat = self.hf_conv(hf)
		hf_attn = self.hf_attn(hf_feat)
		hf_out = hf + hf_feat * hf_attn

		lh_out, hl_out, hh_out = hf_out.chunk(3, dim=1)
		return ll_out, lh_out, hl_out, hh_out


class CrossBranchFusion(nn.Module):
	"""Fuse spatial and wavelet features"""

	def __init__(self, channels):
		super().__init__()
		self.spatial_proj = nn.Conv2d(channels, channels, 1)
		self.wavelet_proj = nn.Conv2d(channels, channels, 1)
		self.gate = nn.Sequential(
			nn.Conv2d(channels * 2, channels, 1),
			nn.Sigmoid(),
		)
		self.out = nn.Conv2d(channels, channels, 1)

	def forward(self, spatial_feat, wavelet_feat):
		if spatial_feat.shape[2:] != wavelet_feat.shape[2:]:
			wavelet_feat = F.interpolate(wavelet_feat, size=spatial_feat.shape[2:], mode="bilinear", align_corners=False)

		s = self.spatial_proj(spatial_feat)
		w = self.wavelet_proj(wavelet_feat)
		gate = self.gate(torch.cat([s, w], dim=1))
		fused = gate * s + (1 - gate) * w
		return self.out(fused)


# ======================== MAIN NETWORK ========================
class WaveFusionNet(nn.Module):
	"""WaveFusion-Net: Dual-branch architecture for image deblurring"""

	def __init__(self, in_channels=3, out_channels=3, base_channels=48, num_blocks=(4, 6, 6, 4)):
		super().__init__()

		self.dwt = DWT()
		self.idwt = IDWT()

		self.intro = nn.Conv2d(in_channels, base_channels, 3, padding=1)

		self.enc1 = nn.Sequential(*[NAFBlock(base_channels) for _ in range(num_blocks[0])])
		self.down1 = nn.Conv2d(base_channels, base_channels * 2, 2, stride=2)

		self.enc2 = nn.Sequential(*[NAFBlock(base_channels * 2) for _ in range(num_blocks[1])])
		self.down2 = nn.Conv2d(base_channels * 2, base_channels * 4, 2, stride=2)

		self.enc3 = nn.Sequential(*[NAFBlock(base_channels * 4) for _ in range(num_blocks[2])])
		self.down3 = nn.Conv2d(base_channels * 4, base_channels * 8, 2, stride=2)

		self.wav_intro = nn.Conv2d(in_channels, base_channels, 3, padding=1)
		self.wav_block1 = WaveletBlock(base_channels)
		self.wav_proj1 = nn.Conv2d(base_channels, base_channels * 2, 1)
		self.wav_block2 = WaveletBlock(base_channels * 2)
		self.wav_proj2 = nn.Conv2d(base_channels * 2, base_channels * 4, 1)
		self.wav_block3 = WaveletBlock(base_channels * 4)

		self.fusion1 = CrossBranchFusion(base_channels * 2)
		self.fusion2 = CrossBranchFusion(base_channels * 4)

		self.bottleneck = nn.Sequential(
			NAFBlock(base_channels * 8),
			StripAttention(base_channels * 8),
			NAFBlock(base_channels * 8),
			NAFBlock(base_channels * 8),
		)

		self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
		self.dec3 = nn.Sequential(*[NAFBlock(base_channels * 4) for _ in range(num_blocks[2])])

		self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
		self.dec2 = nn.Sequential(*[NAFBlock(base_channels * 2) for _ in range(num_blocks[1])])

		self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
		self.dec1 = nn.Sequential(*[NAFBlock(base_channels) for _ in range(num_blocks[0])])

		self.refine = nn.Sequential(
			nn.Conv2d(base_channels, base_channels, 3, padding=1),
			nn.GELU(),
			nn.Conv2d(base_channels, base_channels, 3, padding=1),
		)

		self.outro = nn.Conv2d(base_channels, out_channels, 3, padding=1)

	def forward(self, x):
		B, C, H, W = x.shape

		pad_h = (8 - H % 8) % 8
		pad_w = (8 - W % 8) % 8
		if pad_h > 0 or pad_w > 0:
			x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")

		f0 = self.intro(x)
		f1 = self.enc1(f0)
		f1_down = self.down1(f1)

		f2 = self.enc2(f1_down)
		f2_down = self.down2(f2)

		f3 = self.enc3(f2_down)
		f3_down = self.down3(f3)

		w0 = self.wav_intro(x)

		ll1, lh1, hl1, hh1 = self.dwt(w0)
		ll1, lh1, hl1, hh1 = self.wav_block1(ll1, lh1, hl1, hh1)
		w1 = self.wav_proj1(ll1)

		ll2, lh2, hl2, hh2 = self.dwt(w1)
		ll2, lh2, hl2, hh2 = self.wav_block2(ll2, lh2, hl2, hh2)
		w2 = self.wav_proj2(ll2)

		ll3, lh3, hl3, hh3 = self.dwt(w2)
		ll3, lh3, hl3, hh3 = self.wav_block3(ll3, lh3, hl3, hh3)

		f2_fused = self.fusion1(f2, w1)
		f3_fused = self.fusion2(f3, w2)

		bottleneck_out = self.bottleneck(f3_down)

		d3 = self.up3(bottleneck_out) + f3_fused
		d3 = self.dec3(d3)

		d2 = self.up2(d3) + f2_fused
		d2 = self.dec2(d2)

		d1 = self.up1(d2) + f1
		d1 = self.dec1(d1)

		out = self.refine(d1)
		out = out + f0
		out = self.outro(out)
		out = out + x

		if pad_h > 0 or pad_w > 0:
			out = out[:, :, :H, :W]

		return out


# ======================== LOSS FUNCTIONS ========================
class VGGPerceptualLoss(nn.Module):
	"""VGG-based perceptual loss"""

	def __init__(self):
		super().__init__()
		vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features

		self.slice1 = nn.Sequential(*list(vgg.children())[:4])
		self.slice2 = nn.Sequential(*list(vgg.children())[4:9])
		self.slice3 = nn.Sequential(*list(vgg.children())[9:18])

		for param in self.parameters():
			param.requires_grad = False

		self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
		self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

	def forward(self, pred, target):
		pred = (pred - self.mean) / self.std
		target = (target - self.mean) / self.std

		pred_f1 = self.slice1(pred)
		pred_f2 = self.slice2(pred_f1)
		pred_f3 = self.slice3(pred_f2)

		with torch.no_grad():
			target_f1 = self.slice1(target)
			target_f2 = self.slice2(target_f1)
			target_f3 = self.slice3(target_f2)

		loss = F.l1_loss(pred_f1, target_f1) + F.l1_loss(pred_f2, target_f2) + F.l1_loss(pred_f3, target_f3)
		return loss


class FFTLoss(nn.Module):
	def forward(self, pred, target):
		pred_fft = torch.fft.rfft2(pred)
		target_fft = torch.fft.rfft2(target)
		return F.l1_loss(pred_fft.real, target_fft.real) + F.l1_loss(pred_fft.imag, target_fft.imag)


class GradientLoss(nn.Module):
	def __init__(self):
		super().__init__()
		sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
		sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
		self.register_buffer("sobel_x", sobel_x.view(1, 1, 3, 3).repeat(3, 1, 1, 1))
		self.register_buffer("sobel_y", sobel_y.view(1, 1, 3, 3).repeat(3, 1, 1, 1))

	def forward(self, pred, target):
		pred_gx = F.conv2d(pred, self.sobel_x, padding=1, groups=3)
		pred_gy = F.conv2d(pred, self.sobel_y, padding=1, groups=3)
		target_gx = F.conv2d(target, self.sobel_x, padding=1, groups=3)
		target_gy = F.conv2d(target, self.sobel_y, padding=1, groups=3)
		return F.l1_loss(pred_gx, target_gx) + F.l1_loss(pred_gy, target_gy)


class WaveletHFLoss(nn.Module):
	def __init__(self):
		super().__init__()
		self.dwt = DWT()

	def forward(self, pred, target):
		_, pred_lh, pred_hl, pred_hh = self.dwt(pred)
		_, target_lh, target_hl, target_hh = self.dwt(target)
		return F.l1_loss(pred_lh, target_lh) + F.l1_loss(pred_hl, target_hl) + F.l1_loss(pred_hh, target_hh)


class CombinedLoss(nn.Module):
	def __init__(self):
		super().__init__()
		self.l1 = nn.L1Loss()
		self.vgg = VGGPerceptualLoss()
		self.fft = FFTLoss()
		self.gradient = GradientLoss()
		self.wavelet_hf = WaveletHFLoss()

		self.w_l1 = 1.0
		self.w_vgg = 0.1
		self.w_fft = 0.05
		self.w_gradient = 0.1
		self.w_wavelet = 0.02

	def forward(self, pred, target):
		l1_loss = self.l1(pred, target)
		vgg_loss = self.vgg(pred, target)
		fft_loss = self.fft(pred, target)
		gradient_loss = self.gradient(pred, target)
		wavelet_loss = self.wavelet_hf(pred, target)

		total_loss = (
			self.w_l1 * l1_loss
			+ self.w_vgg * vgg_loss
			+ self.w_fft * fft_loss
			+ self.w_gradient * gradient_loss
			+ self.w_wavelet * wavelet_loss
		)

		return total_loss, {
			"l1": float(l1_loss.detach().cpu().item()),
			"vgg": float(vgg_loss.detach().cpu().item()),
			"fft": float(fft_loss.detach().cpu().item()),
			"gradient": float(gradient_loss.detach().cpu().item()),
			"wavelet": float(wavelet_loss.detach().cpu().item()),
		}


# ======================== DATASET ========================
class RealBlurJ(Dataset):
	"""RealBlur-J dataset loader: root_dir/{train|test}/{blur|gt}/*.png"""

	def __init__(self, root_dir: str, split: str = "train", patch_size: int | None = 256):
		self.root_dir = root_dir
		self.split = split
		self.patch_size = patch_size

		blur_dir = os.path.join(root_dir, split, "blur")
		gt_dir = os.path.join(root_dir, split, "gt")

		self.blur_images: list[str] = []
		self.sharp_images: list[str] = []

		# Support multiple extensions (some mirrors use jpg/jpeg)
		blur_files: list[str] = []
		if os.path.exists(gt_dir) and os.path.exists(blur_dir):
			for ext in ("png", "jpg", "jpeg", "PNG", "JPG", "JPEG"):
				blur_files.extend(glob(os.path.join(blur_dir, f"*.{ext}")))
			blur_files = sorted(blur_files)
		else:
			blur_files = []
		missing = 0
		for blur_path in blur_files:
			filename = os.path.basename(blur_path)
			gt_path = os.path.join(gt_dir, filename)
			if os.path.exists(gt_path):
				self.blur_images.append(blur_path)
				self.sharp_images.append(gt_path)
			else:
				missing += 1

		print(f"Found {len(self.blur_images)} {split} image pairs in RealBlur-J (missing: {missing})")
		if len(self.blur_images) == 0:
			print("[DEBUG] RealBlur-J expected folders:")
			print("        blur_dir:", blur_dir)
			print("        gt_dir  :", gt_dir)

		self.to_tensor = transforms.ToTensor()

	def __len__(self):
		return len(self.blur_images)

	def __getitem__(self, idx):
		blur = Image.open(self.blur_images[idx]).convert("RGB")
		sharp = Image.open(self.sharp_images[idx]).convert("RGB")

		blur = self.to_tensor(blur)
		sharp = self.to_tensor(sharp)

		if self.split == "train" and self.patch_size:
			_, h, w = blur.shape
			if h >= self.patch_size and w >= self.patch_size:
				top = np.random.randint(0, h - self.patch_size + 1)
				left = np.random.randint(0, w - self.patch_size + 1)
				blur = blur[:, top : top + self.patch_size, left : left + self.patch_size]
				sharp = sharp[:, top : top + self.patch_size, left : left + self.patch_size]

			if np.random.random() > 0.5:
				blur = torch.flip(blur, [2])
				sharp = torch.flip(sharp, [2])
			if np.random.random() > 0.5:
				blur = torch.flip(blur, [1])
				sharp = torch.flip(sharp, [1])

		return blur, sharp

	def relpaths(self) -> list[str]:
		"""Returns relative paths for proof/logging."""
		out = []
		for b in self.blur_images:
			# Example: train/blur/001_1.png
			out.append(os.path.join(self.split, "blur", os.path.basename(b)).replace("\\", "/"))
		return out


# ======================== METRICS ========================
def calculate_psnr(pred, target):
	mse = F.mse_loss(pred, target)
	if float(mse.detach().cpu().item()) == 0.0:
		return float("inf")
	return 10 * torch.log10(1.0 / mse)


def calculate_ssim(pred, target, window_size=11):
	C1 = 0.01**2
	C2 = 0.03**2

	sigma = 1.5
	gauss = torch.exp(-torch.arange(window_size, device=pred.device).float().sub(window_size // 2).pow(2) / (2 * sigma**2))
	gauss = gauss / gauss.sum()
	window = gauss.unsqueeze(0) * gauss.unsqueeze(1)
	window = window.unsqueeze(0).unsqueeze(0).expand(3, 1, window_size, window_size)

	mu1 = F.conv2d(pred, window, padding=window_size // 2, groups=3)
	mu2 = F.conv2d(target, window, padding=window_size // 2, groups=3)

	mu1_sq = mu1.pow(2)
	mu2_sq = mu2.pow(2)
	mu1_mu2 = mu1 * mu2

	sigma1_sq = F.conv2d(pred * pred, window, padding=window_size // 2, groups=3) - mu1_sq
	sigma2_sq = F.conv2d(target * target, window, padding=window_size // 2, groups=3) - mu2_sq
	sigma12 = F.conv2d(pred * target, window, padding=window_size // 2, groups=3) - mu1_mu2

	ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
	return ssim_map.mean()


# ======================== VISUALIZATION ========================
def visualize_results(model, test_loader, device, save_dir, num_samples=10):
	print("\n=== Generating Visual Comparisons (TEST) ===")
	model.eval()

	results_dir = os.path.join(save_dir, "results_realblurj")
	os.makedirs(results_dir, exist_ok=True)

	total_samples = len(test_loader.dataset)
	selected_indices = random.sample(range(total_samples), min(num_samples, total_samples))
	selected_indices.sort()
	print(f"Randomly selected {len(selected_indices)} test images for visualization...")

	with torch.no_grad():
		for idx, sample_idx in enumerate(selected_indices):
			blur, sharp = test_loader.dataset[sample_idx]
			blur = blur.unsqueeze(0).to(device)
			sharp = sharp.unsqueeze(0).to(device)
			with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
				pred = model(blur)
			pred = torch.clamp(pred, 0, 1)

			psnr = float(calculate_psnr(pred, sharp).detach().cpu().item())
			ssim = float(calculate_ssim(pred, sharp).detach().cpu().item())

			blur_np = blur.squeeze(0).cpu().numpy().transpose(1, 2, 0)
			pred_np = pred.squeeze(0).cpu().numpy().transpose(1, 2, 0)
			sharp_np = sharp.squeeze(0).cpu().numpy().transpose(1, 2, 0)

			fig, axes = plt.subplots(1, 3, figsize=(15, 5))
			axes[0].imshow(blur_np)
			axes[0].set_title("Blur Input")
			axes[0].axis("off")

			axes[1].imshow(pred_np)
			axes[1].set_title(f"Deblurred\nPSNR: {psnr:.2f} dB | SSIM: {ssim:.4f}")
			axes[1].axis("off")

			axes[2].imshow(sharp_np)
			axes[2].set_title("Ground Truth")
			axes[2].axis("off")

			plt.tight_layout()
			plt.savefig(os.path.join(results_dir, f"sample_{idx}_comparison.png"), dpi=150, bbox_inches="tight")
			plt.close()

	print("\n✓ Saved comparisons to:", results_dir)


# ======================== TRAINING ========================
def train_realblurj():
	config = {
		"data_root": "/kaggle/input/realblurj-image-deblurring-dataset/RealBlur",
		"batch_size": 1,
		"grad_accum_steps": 2,
		"patch_size": 192,
		"epochs": 70,
		"lr": 2e-4,
		"min_lr": 1e-7,
		"num_workers": 0,
		"val_num_workers": 0,
		"test_num_workers": 0,
		"persistent_workers": False,
		"base_channels": 48,
		"num_blocks": (4, 6, 6, 4),
		"save_dir": "/kaggle/working",
		"val_ratio": 0.1,
		"split_seed": 42,
		"cuda_empty_cache_interval": 50,
		"log_cuda_mem_every": 200,
	}

	seed_everything(int(config["split_seed"]))

	# Resolve dataset root robustly (prevents 'Found 0 pairs' when Kaggle mounts under a different name)
	resolved_root = resolve_realblur_root(config["data_root"])
	if resolved_root != config["data_root"]:
		print("[INFO] data_root not found or empty. Auto-detected RealBlur root:")
		print("       ", resolved_root)
	config["data_root"] = resolved_root

	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	print(f"Using device: {device}")
	if device.type == "cuda":
		torch.backends.cudnn.benchmark = False

	model = WaveFusionNet(base_channels=config["base_channels"], num_blocks=config["num_blocks"])

	print("\n" + "="*80)
	print("MODEL ARCHITECTURE: WaveFusion-Net (RealBlur-J)")
	print("="*80)
	if torchinfo_summary is not None:
		try:
			# Keep torchinfo on CPU to avoid GPU memory spikes before training starts.
			summary_report = torchinfo_summary(
				model,
				input_size=(1, 3, int(config["patch_size"]), int(config["patch_size"])),
				col_names=("input_size", "output_size", "num_params", "kernel_size", "mult_adds"),
				col_width=20,
				depth=6,
				verbose=0,
				device="cpu",
			)
			print(summary_report)
		except Exception as e:
			print("[WARNING] torchinfo summary failed:", e)
			print("Falling back to parameter summary.")
	else:
		print("[INFO] torchinfo not installed. Install via `pip install torchinfo` to get full layer summary.")

	total_params = sum(p.numel() for p in model.parameters())
	trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
	print("\n" + "-"*80)
	print(f"Total Parameters: {total_params:,}")
	print(f"Trainable Parameters: {trainable_params:,}")
	print(f"Total (Millions): {total_params/1e6:0.2f}M")
	print("="*80 + "\n")
	print("NOTE: training curves are saved after the full run completes.")
	print("      Output path: /kaggle/working/training_curves/realblurj_training_curves.png")
	if device.type == "cuda":
		torch.cuda.empty_cache()

	if torch.cuda.device_count() > 1:
		print(f"Using {torch.cuda.device_count()} GPUs")
		model = nn.DataParallel(model)
	model = model.to(device)

	criterion = CombinedLoss().to(device)
	optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4)
	scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"], eta_min=config["min_lr"])

	scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

	# Datasets
	# IMPORTANT: TEST split must NOT be used for validation/model selection.
	full_train_aug = RealBlurJ(config["data_root"], split="train", patch_size=config["patch_size"])
	full_train_noaug = RealBlurJ(config["data_root"], split="train", patch_size=None)
	test_dataset = RealBlurJ(config["data_root"], split="test", patch_size=None)

	n_total = len(full_train_aug)
	if n_total == 0:
		raise RuntimeError("No training samples found. Check data_root.")

	rng = np.random.default_rng(int(config["split_seed"]))
	indices = np.arange(n_total)
	rng.shuffle(indices)
	n_val = max(1, int(n_total * float(config["val_ratio"])))
	val_indices = indices[:n_val].tolist()
	train_indices = indices[n_val:].tolist()

	train_dataset = Subset(full_train_aug, train_indices)
	val_dataset = Subset(full_train_noaug, val_indices)

	train_loader = DataLoader(
		train_dataset,
		batch_size=config["batch_size"],
		shuffle=True,
		num_workers=config["num_workers"],
		pin_memory=(device.type == "cuda"),
		drop_last=True,
		persistent_workers=(bool(config["persistent_workers"]) and int(config["num_workers"]) > 0),
	)
	val_loader = DataLoader(
		val_dataset,
		batch_size=1,
		shuffle=False,
		num_workers=int(config["val_num_workers"]),
		pin_memory=(device.type == "cuda"),
		persistent_workers=(bool(config["persistent_workers"]) and int(config["val_num_workers"]) > 0),
	)
	test_loader = DataLoader(
		test_dataset,
		batch_size=1,
		shuffle=False,
		num_workers=int(config["test_num_workers"]),
		pin_memory=(device.type == "cuda"),
		persistent_workers=(bool(config["persistent_workers"]) and int(config["test_num_workers"]) > 0),
	)

	train_losses = []
	val_losses: list[tuple[int, float]] = []
	val_epochs = []
	val_psnrs = []
	val_ssims = []

	# Proof artifact + sanity checks
	train_rel = [full_train_aug.relpaths()[i] for i in train_indices]
	val_rel = [full_train_noaug.relpaths()[i] for i in val_indices]
	test_rel = test_dataset.relpaths()

	overlap_train_val = len(set(train_indices).intersection(set(val_indices)))
	print(f"Split sizes: train={len(train_indices)} val={len(val_indices)} test={len(test_dataset)}")
	print(f"Index overlap train∩val: {overlap_train_val} (should be 0)")

	train_basenames = {os.path.basename(p) for p in train_rel}
	test_basenames = {os.path.basename(p) for p in test_rel}
	basename_overlap = len(train_basenames.intersection(test_basenames))
	print(f"Filename overlap train∩test (basenames only): {basename_overlap} (usually 0)")

	proof_path = export_split_proof(config["save_dir"], train_rel, val_rel, test_rel, config)
	print(f"✓ Split proof written to: {proof_path}")

	# Baseline PSNR check on VAL (blur vs sharp)
	print("\n=== Baseline Check (VAL) ===")
	with torch.no_grad():
		baseline_psnrs = []
		for i, (blur, sharp) in enumerate(val_loader):
			if i >= 5:
				break
			baseline_psnrs.append(float(calculate_psnr(blur, sharp).detach().cpu().item()))
		print(f"Baseline PSNR (blur vs sharp): {np.mean(baseline_psnrs):.2f} dB")

	best_val_psnr = -1e9
	best_val_ssim = 0.0

	print("\n=== Starting Training (VAL controls best checkpoint) ===")
	for epoch in range(int(config["epochs"])):
		model.train()
		epoch_loss = 0.0
		loss_components = {"l1": 0.0, "vgg": 0.0, "fft": 0.0, "gradient": 0.0, "wavelet": 0.0}
		accum_steps = max(1, int(config.get("grad_accum_steps", 1)))
		optimizer.zero_grad(set_to_none=True)

		pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config['epochs']}")
		for step, (blur, sharp) in enumerate(pbar, start=1):
			blur = blur.to(device, non_blocking=True)
			sharp = sharp.to(device, non_blocking=True)

			with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
				pred = model(blur)
				loss, components = criterion(pred, sharp)
			loss_value = float(loss.detach().cpu().item())
			loss = loss / accum_steps

			scaler.scale(loss).backward()

			if step % accum_steps == 0 or step == len(train_loader):
				scaler.unscale_(optimizer)
				torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
				scaler.step(optimizer)
				scaler.update()
				optimizer.zero_grad(set_to_none=True)

			epoch_loss += loss_value
			for k, v in components.items():
				loss_components[k] += float(v)

			postfix = {"loss": f"{loss_value:.4f}", "lr": f"{optimizer.param_groups[0]['lr']:.2e}"}
			if device.type == "cuda" and int(config["log_cuda_mem_every"]) > 0 and step % int(config["log_cuda_mem_every"]) == 0:
				alloc_gb = torch.cuda.memory_allocated(device=device) / (1024**3)
				reserved_gb = torch.cuda.memory_reserved(device=device) / (1024**3)
				postfix["mem"] = f"{alloc_gb:.2f}/{reserved_gb:.2f}GB"
			pbar.set_postfix(postfix)

			del pred, loss
			del blur, sharp
			if device.type == "cuda" and int(config["cuda_empty_cache_interval"]) > 0 and step % int(config["cuda_empty_cache_interval"]) == 0:
				torch.cuda.empty_cache()
				gc.collect()

		scheduler.step()

		n_batches = max(1, len(train_loader))
		epoch_loss /= n_batches
		for k in loss_components:
			loss_components[k] /= n_batches
		train_losses.append(epoch_loss)

		# Validate on VAL only (including final epoch)
		is_final_epoch = (epoch + 1) == int(config["epochs"])
		if (epoch + 1) % 10 == 0 or epoch == 0 or is_final_epoch:
			model.eval()
			val_loss = 0.0
			val_psnr = 0.0
			val_ssim = 0.0
			with torch.no_grad():
				for blur, sharp in tqdm(val_loader, desc="Validating (VAL)"):
					blur = blur.to(device)
					sharp = sharp.to(device)
					with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
						pred = model(blur)
						v_loss, _ = criterion(pred, sharp)
					pred = torch.clamp(pred, 0, 1)
					val_loss += float(v_loss.detach().cpu().item())
					val_psnr += float(calculate_psnr(pred, sharp).detach().cpu().item())
					val_ssim += float(calculate_ssim(pred, sharp).detach().cpu().item())

			val_loss /= len(val_loader)
			val_psnr /= len(val_loader)
			val_ssim /= len(val_loader)
			val_losses.append((epoch + 1, val_loss))
			val_epochs.append(epoch + 1)
			val_psnrs.append(val_psnr)
			val_ssims.append(val_ssim)

			print(f"\nEpoch {epoch + 1}: Loss={epoch_loss:.4f}, VAL Loss={val_loss:.4f}, VAL PSNR={val_psnr:.2f}dB, VAL SSIM={val_ssim:.4f}")
			print(
				f"  Components - L1:{loss_components['l1']:.4f}, VGG:{loss_components['vgg']:.4f}, "
				f"FFT:{loss_components['fft']:.4f}, Grad:{loss_components['gradient']:.4f}, "
				f"Wav:{loss_components['wavelet']:.4f}"
			)

			if val_psnr > best_val_psnr:
				best_val_psnr = val_psnr
				best_val_ssim = val_ssim
				save_path = os.path.join(config["save_dir"], "best_model_realblurj.pth")
				state_dict = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
				torch.save(
					{
						"epoch": epoch + 1,
						"model_state_dict": state_dict,
						"optimizer_state_dict": optimizer.state_dict(),
						"val_psnr": val_psnr,
						"val_ssim": val_ssim,
						"split_proof": os.path.basename(proof_path),
					},
					save_path,
				)
				print(f"  *** New best model saved (by VAL)! VAL PSNR: {val_psnr:.2f}dB ***")

			if device.type == "cuda":
				torch.cuda.empty_cache()
				gc.collect()

		if (epoch + 1) % 20 == 0:
			save_path = os.path.join(config["save_dir"], f"checkpoint_realblurj_epoch{epoch + 1}.pth")
			state_dict = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
			torch.save(
				{
					"epoch": epoch + 1,
					"model_state_dict": state_dict,
					"optimizer_state_dict": optimizer.state_dict(),
					"scheduler_state_dict": scheduler.state_dict(),
				},
				save_path,
			)

	print("\n=== Training Complete ===")
	print(f"Best VAL PSNR: {best_val_psnr:.2f}dB | Best VAL SSIM: {best_val_ssim:.4f}")

	# Final TEST evaluation (single shot) using best VAL checkpoint
	print("\n=== Final Test Evaluation (TEST) ===")
	best_model_path = os.path.join(config["save_dir"], "best_model_realblurj.pth")
	if os.path.exists(best_model_path):
		checkpoint = torch.load(best_model_path, map_location=device)
		state = checkpoint.get("model_state_dict", checkpoint)
		if hasattr(model, "module"):
			model.module.load_state_dict(state)
		else:
			model.load_state_dict(state)
		print(f"Loaded best-VAL model from epoch {checkpoint.get('epoch', 'N/A')}")
	else:
		print("Warning: best model not found; evaluating current weights.")

	model.eval()
	test_psnr = 0.0
	test_ssim = 0.0
	if device.type == "cuda":
		torch.cuda.empty_cache()
		gc.collect()
	with torch.no_grad():
		for blur, sharp in tqdm(test_loader, desc="Testing (TEST)"):
			blur = blur.to(device)
			sharp = sharp.to(device)
			with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
				pred = model(blur)
			pred = torch.clamp(pred, 0, 1)
			test_psnr += float(calculate_psnr(pred, sharp).detach().cpu().item())
			test_ssim += float(calculate_ssim(pred, sharp).detach().cpu().item())

	test_psnr /= len(test_loader)
	test_ssim /= len(test_loader)
	print(f"TEST PSNR={test_psnr:.2f}dB, TEST SSIM={test_ssim:.4f}")

	# Save training curves (Train Loss & Val Loss)
	curve_dir = os.path.join(config["save_dir"], "training_curves")
	os.makedirs(curve_dir, exist_ok=True)
	curve_path = os.path.join(curve_dir, "realblurj_training_curves.png")
	fig, ax = plt.subplots(figsize=(8, 5))
	ax.plot(range(1, len(train_losses) + 1), train_losses, color="#1f77b4", linewidth=1.5, label="Train Loss")
	if val_losses:
		v_epochs, v_vals = zip(*val_losses)
		ax.plot(v_epochs, v_vals, color="#ff7f0e", linewidth=1.5, label="Val Loss")
	ax.set_xlabel("Epoch", fontsize=12)
	ax.set_ylabel("Loss", fontsize=12)
	ax.set_title("Training & Validation Loss Curve", fontsize=13)
	ax.legend(fontsize=11)
	ax.grid(True, alpha=0.3)
	ax.set_xlim(left=0)
	plt.tight_layout()
	plt.savefig(curve_path, dpi=150, bbox_inches="tight")
	plt.close(fig)
	print(f"✓ Training curves saved to: {curve_path}")

	# Save VAL PSNR & VAL SSIM curve (twin y-axis)
	if val_epochs and val_psnrs and val_ssims:
		val_metrics_path = os.path.join(curve_dir, "realblurj_val_metrics.png")
		fig2, ax_psnr = plt.subplots(figsize=(8, 5))
		ax_ssim = ax_psnr.twinx()

		ax_psnr.plot(val_epochs, val_psnrs, color="#2ca02c", linewidth=1.5, label="Val PSNR (dB)")
		ax_ssim.plot(val_epochs, val_ssims, color="#d62728", linewidth=1.5, linestyle="--", label="Val SSIM")

		ax_psnr.set_xlabel("Epoch", fontsize=12)
		ax_psnr.set_ylabel("PSNR (dB)", color="#2ca02c", fontsize=12)
		ax_ssim.set_ylabel("SSIM", color="#d62728", fontsize=12)
		ax_psnr.set_title("Validation PSNR & SSIM Curve", fontsize=13)

		lines1, labels1 = ax_psnr.get_legend_handles_labels()
		lines2, labels2 = ax_ssim.get_legend_handles_labels()
		ax_psnr.legend(lines1 + lines2, labels1 + labels2, fontsize=11)

		ax_psnr.grid(True, alpha=0.3)
		ax_psnr.set_xlim(left=0)
		plt.tight_layout()
		plt.savefig(val_metrics_path, dpi=150, bbox_inches="tight")
		plt.close(fig2)
		print(f"✓ Validation metrics curve saved to: {val_metrics_path}")

	# Optional: visualize
	print("\nLoading best model for visualization...")
	if os.path.exists(best_model_path):
		checkpoint = torch.load(best_model_path, map_location=device)
		if hasattr(model, "module"):
			model.module.load_state_dict(checkpoint["model_state_dict"])
		else:
			model.load_state_dict(checkpoint["model_state_dict"])
		visualize_results(model, test_loader, device, config["save_dir"], num_samples=10)
	else:
		print("Warning: Best model not found. Skipping visualization.")

	return test_psnr, test_ssim


if __name__ == "__main__":
	train_realblurj()



[INFO] data_root not found or empty. Auto-detected RealBlur root:
        /kaggle/input/datasets/adwythdarsanr/realblurj-image-deblurring-dataset/RealBlur
Using device: cuda

MODEL ARCHITECTURE: WaveFusion-Net (RealBlur-J)
Layer (type:depth-idx)                        Input Shape          Output Shape         Param #              Kernel Shape         Mult-Adds
WaveFusionNet                                 [1, 3, 192, 192]     [1, 3, 192, 192]     --                   --                   --
├─Conv2d: 1-1                                 [1, 3, 192, 192]     [1, 48, 192, 192]    1,344                [3, 3]               49,545,216
├─Sequential: 1-2                             [1, 48, 192, 192]    [1, 48, 192, 192]    --                   --                   --
│    └─NAFBlock: 2-1                          [1, 48, 192, 192]    [1, 48, 192, 192]    --                   --                   --
│    │    └─LayerNorm: 3-1                    [1, 192, 192, 48]    [1, 192, 192, 48]    96       

100%|██████████| 548M/548M [00:02<00:00, 239MB/s] 


Found 3758 train image pairs in RealBlur-J (missing: 0)
Found 3758 train image pairs in RealBlur-J (missing: 0)
Found 980 test image pairs in RealBlur-J (missing: 0)
Split sizes: train=3383 val=375 test=980
Index overlap train∩val: 0 (should be 0)
Filename overlap train∩test (basenames only): 0 (usually 0)
✓ Split proof written to: /kaggle/working/split_proof_realblurj.json

=== Baseline Check (VAL) ===
Baseline PSNR (blur vs sharp): 27.26 dB

=== Starting Training (VAL controls best checkpoint) ===


Validating (VAL): 100%|██████████| 375/375 [02:43<00:00,  2.29it/s]



Epoch 1: Loss=0.4184, VAL Loss=1.0542, VAL PSNR=27.30dB, VAL SSIM=0.8010
  Components - L1:0.0300, VGG:1.3314, FFT:4.7945, Grad:0.1494, Wav:0.0313
  *** New best model saved (by VAL)! VAL PSNR: 27.30dB ***


...


Epoch 60: Loss=0.3112, VAL Loss=0.8496, VAL PSNR=30.07dB, VAL SSIM=0.8871
  Components - L1:0.0200, VGG:0.9948, FFT:3.6090, Grad:0.1072, Wav:0.0254
  *** New best model saved (by VAL)! VAL PSNR: 30.07dB ***


Validating (VAL): 100%|██████████| 375/375 [02:28<00:00,  2.52it/s]



Epoch 70: Loss=0.3098, VAL Loss=0.8481, VAL PSNR=30.08dB, VAL SSIM=0.8877
  Components - L1:0.0198, VGG:0.9904, FFT:3.5977, Grad:0.1064, Wav:0.0253
  *** New best model saved (by VAL)! VAL PSNR: 30.08dB ***

=== Training Complete ===
Best VAL PSNR: 30.08dB | Best VAL SSIM: 0.8877

=== Final Test Evaluation (TEST) ===
Loaded best-VAL model from epoch 70


Testing (TEST): 100%|██████████| 980/980 [06:36<00:00,  2.47it/s]


TEST PSNR=29.57dB, TEST SSIM=0.8839
✓ Training curves saved to: /kaggle/working/training_curves/realblurj_training_curves.png
✓ Validation metrics curve saved to: /kaggle/working/training_curves/realblurj_val_metrics.png

Loading best model for visualization...

=== Generating Visual Comparisons (TEST) ===
Randomly selected 10 test images for visualization...

✓ Saved comparisons to: /kaggle/working/results_realblurj
